# 00 · 评价指标 Demo

实现并验证 PSNR 与 SSIM；无需真实数据，随机张量即可跑通。

In [ ]:
import torch
import torch.nn.functional as F


def psnr(pred, gt, max_val=1.0):
    mse = F.mse_loss(pred, gt, reduction="mean")
    if mse == 0:
        return torch.tensor(99.0)
    return 10.0 * torch.log10(max_val ** 2 / mse)


def _gaussian_window(window_size, sigma, channels, device):
    coords = torch.arange(window_size, dtype=torch.float32, device=device)
    coords -= window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).unsqueeze(1)
    w2d = (g @ g.t()).unsqueeze(0).unsqueeze(0)
    return w2d.expand(channels, 1, window_size, window_size).contiguous()


def ssim(pred, gt, window_size=11, max_val=1.0):
    channels = pred.shape[1]
    window = _gaussian_window(window_size, 1.5, channels, pred.device)
    pad = window_size // 2
    mu_x = F.conv2d(pred, window, padding=pad, groups=channels)
    mu_y = F.conv2d(gt, window, padding=pad, groups=channels)
    sigma_x2 = F.conv2d(pred ** 2, window, padding=pad, groups=channels) - mu_x ** 2
    sigma_y2 = F.conv2d(gt ** 2, window, padding=pad, groups=channels) - mu_y ** 2
    sigma_xy = F.conv2d(pred * gt, window, padding=pad, groups=channels) - mu_x * mu_y
    C1, C2 = (0.01 * max_val) ** 2, (0.03 * max_val) ** 2
    s = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / (
        (mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x2 + sigma_y2 + C2))
    return s.mean()


In [ ]:
# 随机张量验证：相同图 PSNR 应接近 99，SSIM 接近 1
gt = torch.rand(1, 3, 256, 256)
noise = (gt + 0.02 * torch.randn_like(gt)).clamp(0, 1)
print("PSNR(gt, gt) =", psnr(gt, gt).item())
print("PSNR(noise)  =", psnr(noise, gt).item())
print("SSIM(noise)  =", ssim(noise, gt).item())
